## Index městské dostupnosti vážený populací
### Workflow výpočtu indexu pro parcely z katastru nemovitostí
Tento notebook vypočítá index pro vložené soubory za použití potřebných knihoven.

### Vzorec pro výpočet indexu (PUAI)
$PUAI_j = \sum_{i=1}^{n} P_i e^{-\beta d_{ij}}$

- P_i: počet obyvatel města i
- d_ij: nejkratší vzdálenost v kilometrech ze středu parcely j do města i
- beta: parametr exponenciálního poklesu vlivu vzdálenosti (v km⁻¹)

Pro účely testování je beta nastavena na 0.05.

### Vstupní data
Soubor dostupnost_data.gpkg obsahující 2 vrstvy:
- parcely - polygonová vrstva katastrálních parcel obsahující unikátní ID
- mesta - bodová vrstva měst obsahující unikátní ID a atribut pocet_obyv

### Důležité
Pro výpočet musí být zvolen vhodný souřadnicový systém.

## Instalace a import balíčků
Pokud je to potřeba odkomentovat řádek s instalací.

In [22]:
# %pip install geopandas pandas numpy matplotlib pyogrio shapely

from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from google.colab import files

uploaded = files.upload()

print("Geopandas:", gpd.__version__)
print("Pandas:", pd.__version__)

Saving dostupnost_data.gpkg to dostupnost_data (3).gpkg
Geopandas: 1.2.0
Pandas: 2.2.3


## Nastavení cest, názvů souborů a parametrů modelu
Nastav hodnoty v následující buňce pro přizpůsobení modelu
- parcely: polygonová vrstva parcel
- mesta: bodová vrstva měst
- CRS: souřadnicový systém (pro ČR ideální EPSG:5514 (S-JTSK / Krovak East North))
- beta: koeficient poklesu vlivu
- popu_exponent: základní nastavení 1.0

In [23]:
# nahraj soubory
data_path = Path("/content/dostupnost_data.gpkg")

parcely_vrstva = "parcely"
mesta_vrstva = "mesta"

# názvy atributů, možno změnit
parcely_id = "id"
mesta_id = "ID"
mesta_nazev = "NAZEV"
mesta_popu = "POPU"

# souřadnicový systém
crs = "EPSG:5514"

# parametry modelu
beta = 0.05
popu_exponent = 1.0

# maximální vzdálenost
max_vzdalenost_km = None

# výstup
output_dir = Path("index_dostupnosti")
output_dir.mkdir(parents=True, exist_ok=True)

print(mesta.geom_type.value_counts())

MultiPoint    421
Name: count, dtype: int64


## Načtení a validacde vstupních dat
Parcely musí být definovány polygonem a města bodem.

In [26]:
def nacist_vektor(path, layer=None):
    if not path.exists():
        raise FileNotFoundError(f"Nemohu najít: {path.resolve()}")
    if layer is None:
        return gpd.read_file(path)
    return gpd.read_file(path, layer=layer)

parcely = nacist_vektor(data_path, parcely_vrstva)
mesta = nacist_vektor(data_path, mesta_vrstva)

print("Parcely:", len(parcely), "| geometrie:", parcely.geom_type.unique())
print("Města:", len(mesta), "| geometry:", mesta.geom_type.unique())
print("CRS parcely:", parcely.crs)
print("CRS města:", mesta.crs)

if parcely.crs is None or mesta.crs is None:
    raise ValueError("Obě vrstvy potřebují definovat CRS před reprojekcí.")

if parcely_id not in parcely.columns:
    raise KeyError(f"'{parcely_id}' nenalezeno.")
if mesta_id not in mesta.columns:
    raise KeyError(f"'{mesta_id}' nenalezeno.")
if mesta_popu not in mesta.columns:
    raise KeyError(f"'{mesta_popu}' nenalezeno")

if parcely[parcely_id].isna().any() or parcely[parcely_id].duplicated().any():
    raise ValueError("ID parcely musí být unikátní a nenulové.")
if mesta[mesta_id].isna().any() or mesta[mesta_id].duplicated().any():
    raise ValueError("ID města musí být unikátní a nenulové.")

mesta[mesta_popu] = pd.to_numeric(mesta[mesta_popu], errors="coerce")
if mesta[mesta_popu].isna().any() or (mesta[mesta_popu] < 0).any():
    raise ValueError("Počet obyvatel musí obsahovat validní, nezáporné hodnoty.")

if not parcely.geom_type.isin(["Polygon", "MultiPolygon"]).all():
    raise ValueError("Vrstva parcel musí obsahovat polygonové nebo multipolygonové geometrie.")
if not mesta.geom_type.eq("MultiPoint").all():
    raise ValueError("Vrstva měst musí obsahovat bodové geometrie.")

print("Validace byla úspěšně provedena.")

Parcely: 10 | geometrie: ['Polygon']
Města: 421 | geometry: ['MultiPoint']
CRS parcely: EPSG:5514
CRS města: EPSG:5514
Validace byla úspěšně provedena.


## Změna souřadnicového systému a vytvoření bodů z parcel

Při výpočtu vzdálenosti se používají těžiště pozemků. U nepravidelných nebo konkávních pozemků může těžiště ležet mimo jejich polygon; funkce `representative_point()` zaručuje, že bod bude ležet uvnitř každého pozemku. Tento notebook ve výchozím nastavení používá reprezentativní body. Pokud vaše metodika výslovně vyžaduje geometrická těžiště, přepněte na tuto možnost.

In [27]:
parcely_crs = parcely.to_crs(crs).copy()
mesta_crs = mesta.to_crs(crs).copy()

# bod musí být uprostřed polygonu
parcely_body = parcely_crs[[parcely_id, "geometry"]].copy()
parcely_body["geometry"] = parcely_body.geometry.representative_point()

# kontrola užívání metrického systému
axis_units = [axis.unit_name.lower() for axis in parcely_body.crs.axis_info]
print("CRS jednotky:", axis_units)
if not all("metre" in unit or "meter" in unit for unit in axis_units):
    raise ValueError("Zvolený CRS nepoužívá metry. Vyberte jiný vhodný.")

parcely_body.to_file(output_dir / "parcely_body.gpkg", driver="GPKG")
print("Vytvořeno bodů parcel:", len(parcely_body))

CRS jednotky: ['metre', 'metre']
Vytvořeno bodů parcel: 10


## Vytvoření matice vzdáleností mezi pozemky a městy
Tím se vytvoří jeden řádek pro každou dvojici „parcela–město“. Při 1000 parcelách a 200 městech bude matice obsahovat 200 000 řádků.

Vzdálenosti se počítají v promítnutém CRS (v metrech) a poté se převádějí na kilometry. Je-li nastavena hodnota `max_vzdalenost_km`, jsou dvojice přesahující tuto prahovou hodnotu vyloučeny. Tato prahová hodnota by měla být odůvodněna a otestována; ponechání hodnoty `None` zahrnuje všechna města.

In [30]:
# GeoPandas/Shapely operace; iterují parcely, aby se zamezilo vytváčení velkému množství dočasných geometrií.

mesta_geometrie = mesta_crs.geometry.to_numpy()
mesta_id_hodnoty = mesta_crs[mesta_id].to_numpy()

vzdalenost_zaznamy = []
for _, parcela in parcely_body.iterrows():
    vzdalenost_m = np.array([parcela.geometry.distance(mesta_geometrie) for mesta_geometrie in mesta_geometrie], dtype=float)
    vzdalenost_km = vzdalenost_m / 1000.0

    for mesto_id, vzdalenost_m, vzdalenost_km in zip(mesta_id_hodnoty, vzdalenost_m, vzdalenost_km):
        if max_vzdalenost_km is None or vzdalenost_km <= max_vzdalenost_km:
            vzdalenost_zaznamy.append({
                parcely_id: parcela[parcely_id],
                mesta_id: mesto_id,
                "vzdalenost_m": vzdalenost_m,
                "vzdalenost_km": vzdalenost_km
            })

vzdalenosti = pd.DataFrame(vzdalenost_zaznamy)

if vzdalenosti.empty:
    raise ValueError("Žádné záznamy, zkontroluj nastavení maximální vzdálenosti")

print("Dvojice parcela-město:", len(vzdalenosti))
display(vzdalenosti.head())

Dvojice parcela-město: 4210


,id,ID,vzdalenost_m,vzdalenost_km
0,1,1,180356.940101,180.356940
1,1,2,294134.993151,294.134993
2,1,3,162245.306693,162.245307
3,1,4,58981.715145,58.981715
4,1,5,285859.330697,285.859331


## Výpočet podílu každého města

In [31]:
mesta_atributy = mesta_crs[[mesta_id, mesta_nazev, mesta_popu]].copy()

# Spoj populaci a název města ke každému záznamu vzdálenosti.
zaznamy = vzdalenosti.merge(
    mesta_atributy,
    on=mesta_id,
    how="left",
    validate="many_to_one"
)

if zaznamy[mesta_popu].isna().any():
    raise ValueError("Některá ID měst nedpovídají tabulce počtu pobyvatel.")

zaznamy["vliv"] = (
    zaznamy[mesta_popu].astype(float) ** popu_exponent
    * np.exp(-beta * zaznamy["vzdalenost_km"])
)

display(zaznamy.head())
zaznamy.to_csv(output_dir / "parcely_mesta_vliv.csv", index=False)

,id,ID,vzdalenost_m,vzdalenost_km,NAZEV,POPU,vliv
0,1,1,180356.940101,180.356940,Adamov,4564,0.553279
1,1,2,294134.993151,294.134993,Albrechtice,3782,0.001551
2,1,3,162245.306693,162.245307,AŠ,12683,3.802856
3,1,4,58981.715145,58.981715,Bakov nad Jizerou,5355,280.535486
4,1,5,285859.330697,285.859331,Baška,4056,0.002516


## Sloučit příspěvky jednotlivých měst do jednoho indexu pro každou parcelu

In [32]:
index_parcely = (
    zaznamy.groupby(parcely_id, as_index=False)
         .agg(
             PUAI_abs=("vliv", "sum"),
             mesta_zahrnuta=(mesta_id, "nunique"),
             nejblizsi_mesto_km=("vzdalenost_km", "min")
         )
)

print("Počet parcel s vypočítaným indexem:", len(index_parcely))
display(index_parcely.head())

index_parcely.to_csv(output_dir / "parcely_index_dostupnosti.csv", index=False)

Počet parcel s vypočítaným indexem: 10


,id,PUAI_abs,mesta_zahrnuta,nejblizsi_mesto_km
0,1,1.423518e+06,421,2.136554
1,10,1.433255e+05,421,2.414279
2,2,4.536427e+05,421,2.949876
3,3,4.900105e+05,421,2.435388
4,4,2.717547e+05,421,1.724083


## Připojení indexu zpět k původním polygonům pozemku

In [33]:
parcely_index = parcely_crs.merge(
    index_parcely,
    on=parcely_id,
    how="left",
    validate="one_to_one"
)

chybi_index = parcely_index["PUAI_abs"].isna().sum()
print("Počet parcel bez indexu:", chybi_index)

if chybi_index:
    print("ID parcel bez indexu:")
    display(parcely_index.loc[parcely_index["PUAI_abs"].isna(), [parcely_id]])

parcely_index.to_file(
    output_dir / "parcely_index_dostupnosti.gpkg",
    layer="parcely_index_dostupnosti",
    driver="GPKG"
)

display(parcely_index[[parcely_id, "PUAI_abs", "mesta_zahrnuta", "nejblizsi_mesto_km"]].head())

Počet parcel bez indexu: 0


,id,PUAI_abs,mesta_zahrnuta,nejblizsi_mesto_km
0,1,1.423518e+06,421,2.136554
1,2,4.536427e+05,421,2.949876
2,3,4.900105e+05,421,2.435388
3,4,2.717547e+05,421,1.724083
4,5,2.379989e+05,421,1.971327


## Normalizovaný index 0 - 10

In [34]:
platne = parcely_index["PUAI_abs"].notna()
if platne.sum() == 0:
    raise ValueError("Žádný platná PUAI hodnota.")

minimum = parcely_index.loc[platne, "PUAI_abs"].min()
maximum = parcely_index.loc[platne, "PUAI_abs"].max()

if np.isclose(maximum, minimum):
    parcely_index["PUAI_10"] = np.where(platne, 0.0, np.nan)
else:
    parcely_index["PUAI_10"] = np.nan
    parcely_index.loc[platne, "PUAI_10"] = (
        10 * (parcely_index.loc[platne, "PUAI_abs"] - minimum) / (maximum - minimum)
    )

final_path = output_dir / "parcely_PUAI.gpkg"
parcely_index.to_file(final_path, layer="parcely_PUAI", driver="GPKG")

print("Uloženo:", final_path.resolve())
display(
    parcely_index[[parcely_id, "PUAI_abs", "PUAI_10", "mesta_zahrnuta", "nejblizsi_mesto_km"]]
    .sort_values("PUAI_10", ascending=False)
    .head(10)
)

Uloženo: /content/index_dostupnosti/parcely_PUAI.gpkg


,id,PUAI_abs,PUAI_10,mesta_zahrnuta,nejblizsi_mesto_km
0,1,1.423518e+06,10.000000,421,2.136554
2,3,4.900105e+05,2.708069,421,2.435388
1,2,4.536427e+05,2.423988,421,2.949876
3,4,2.717547e+05,1.003202,421,1.724083
5,6,2.606858e+05,0.916739,421,0.772466
4,5,2.379989e+05,0.739525,421,1.971327
7,8,2.353139e+05,0.718551,421,2.055232
8,9,2.183159e+05,0.585774,421,4.012906
6,7,1.455478e+05,0.017359,421,2.157480
9,10,1.433255e+05,0.000000,421,2.414279
